# 🔬 BugBuster — Test Dashboard

Interactive hardware-in-the-loop test runner for the BugBuster I/O board.

| Step | Action |
|------|--------|
| **1** | Run **Cell 2** (Setup) |
| **2** | Run **Cell 3** (Build UI) — fill in connection details |
| **3** | Run **Cell 4** (Wire runner) |
| **4** | Click **▶ Run Tests** |

> **Requirements:** activate the project venv first: `source .venv/bin/activate`

In [ ]:
"""
Environment check — run this first.
Verifies you're using the repo .venv and shows how to fix it if not.
"""
import sys, subprocess
from pathlib import Path
from IPython.display import display, HTML

# Locate repo root (same logic as setup cell)
_here = Path.cwd()
REPO_ROOT = None
for _p in [_here, _here.parent, _here.parent.parent, _here.parent.parent.parent]:
    if (_p / "tests" / "pytest.ini").exists():
        REPO_ROOT = _p.resolve()
        break
if REPO_ROOT is None:
    REPO_ROOT = _here.parent.resolve()

VENV_DIR    = REPO_ROOT / ".venv"
VENV_PYTHON = VENV_DIR / "bin" / "python"
CURRENT_PY  = Path(sys.executable).resolve()
IN_VENV     = str(CURRENT_PY).startswith(str(VENV_DIR))

print(f"Repo root     : {REPO_ROOT}")
print(f"Current Python: {sys.executable}")
print(f"Expected venv : {VENV_PYTHON}")
print()

if IN_VENV:
    display(HTML("""
    <div style="background:#0d2818;border:1px solid #3fb950;border-radius:8px;
                padding:10px 16px;font-family:monospace;font-size:13px">
      <b style="color:#3fb950">✅ Correct kernel — running inside .venv</b><br>
      <span style="color:#8b949e">All imports and test runs will use the repo virtual environment.</span>
    </div>
    """))
else:
    venv_exists = VENV_DIR.exists()

    if not venv_exists:
        display(HTML(f"""
        <div style="background:#1c0a0a;border:1px solid #f85149;border-radius:8px;
                    padding:12px 16px;font-family:monospace;font-size:13px">
          <b style="color:#f85149">❌ .venv not found at {VENV_DIR}</b><br><br>
          <b style="color:#ccc">Run setup first (once, in a terminal):</b><br>
          <code style="color:#58a6ff">bash {REPO_ROOT}/tests/setup.sh</code><br><br>
          <span style="color:#8b949e">Then come back here and follow the kernel-selection steps below.</span>
        </div>
        """))
    else:
        display(HTML(f"""
        <div style="background:#1c1407;border:1px solid #d29922;border-radius:8px;
                    padding:12px 16px;font-family:monospace;font-size:13px">
          <b style="color:#d29922">⚠️ Wrong kernel — not using .venv</b><br>
          <span style="color:#8b949e">Current: <code>{sys.executable}</code></span><br><br>
          <b style="color:#e6edf3">Fix in VS Code:</b><br>
          &nbsp; 1. Click the kernel name in the top-right corner of this notebook<br>
          &nbsp; 2. Select <b style="color:#58a6ff">Python Environments…</b><br>
          &nbsp; 3. Pick <b style="color:#58a6ff">.venv</b>
              &nbsp;<span style="color:#8b949e">(at <code>{VENV_DIR}</code>)</span><br>
        </div>
        """))

        # Try to auto-register the kernel so it shows up in VS Code
        reg = subprocess.run(
            [str(VENV_PYTHON), "-m", "ipykernel", "install",
             "--user", "--name=bugbuster",
             "--display-name=BugBuster (.venv)"],
            capture_output=True, text=True,
        )
        if reg.returncode == 0:
            display(HTML("""
            <div style="background:#0d2818;border:1px solid #3fb950;border-radius:6px;
                        padding:8px 14px;margin-top:8px;font-family:monospace;font-size:12px">
              <b style="color:#3fb950">✅ Kernel registered as “BugBuster (.venv)”</b><br>
              <span style="color:#8b949e">
                Reload VS Code
                <code style="color:#ccc">Cmd+Shift+P → Developer: Reload Window</code>,
                then select <b>BugBuster (.venv)</b> from the kernel picker and
                re-run this cell to confirm.
              </span>
            </div>
            """))
        else:
            display(HTML(f"""
            <div style="background:#1c1407;border:1px solid #d29922;border-radius:6px;
                        padding:8px 14px;margin-top:8px;font-family:monospace;font-size:12px">
              <b style="color:#d29922">Could not auto-register kernel.</b>
              Run manually in a terminal:<br>
              <code style="color:#58a6ff">source {VENV_DIR}/bin/activate && pip install ipykernel</code><br>
              <code style="color:#58a6ff">python -m ipykernel install --user --name=bugbuster
                  --display-name="BugBuster (.venv)"</code>
            </div>
            """))


In [ ]:
"""
Setup cell — run once per kernel session.
Finds repo root, installs missing packages, detects serial ports.
"""
import sys, subprocess, json, os, threading, time, re
import glob as _glob
from pathlib import Path

# ── Find repo root ──────────────────────────────────────────────────────────
_here = Path.cwd()
REPO_ROOT = None
for _p in [_here, _here.parent, _here.parent.parent, _here.parent.parent.parent]:
    if (_p / "tests" / "pytest.ini").exists():
        REPO_ROOT = _p.resolve()
        break
if REPO_ROOT is None:
    REPO_ROOT = _here.parent.resolve()
    print(f"⚠  Could not auto-detect repo root. Using: {REPO_ROOT}")

TESTS_DIR = REPO_ROOT / "tests"
sys.path.insert(0, str(REPO_ROOT / "python"))

# ── Ensure packages ─────────────────────────────────────────────────────────
def _ensure(import_name, pip_name=None):
    try:
        __import__(import_name)
    except ImportError:
        pkg = pip_name or import_name
        print(f"  Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Checking dependencies...")
_ensure("ipywidgets")
_ensure("plotly")
_ensure("pandas")
_ensure("pytest_jsonreport", "pytest-json-report")
_ensure("pytest_timeout",    "pytest-timeout")
_ensure("pytest_html",       "pytest-html")

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd

# ── Serial port detection ────────────────────────────────────────────────────
def detect_serial_ports():
    patterns = ["/dev/cu.usbmodem*", "/dev/cu.usbserial*",
                "/dev/ttyACM*", "/dev/ttyUSB*", "COM*"]
    ports = []
    for pat in patterns:
        ports.extend(_glob.glob(pat))
    return sorted(set(ports))

_ports = detect_serial_ports()
print(f"✅  Setup complete")
print(f"    Repo:   {REPO_ROOT}")
print(f"    Tests:  {TESTS_DIR}")
print(f"    Ports:  {_ports or ['none detected']}")

In [ ]:
"""
Build UI cell — creates all widgets and assembles the dashboard layout.
Run this cell to display the interactive test dashboard.
"""

# ── Category definitions ─────────────────────────────────────────────────────
CATEGORIES = [
    ("core",      "01", "Core device / firmware info"),
    ("channels",  "02", "Channel configuration (AD74416H)"),
    ("gpio",      "03", "GPIO pins A\u2013F"),
    ("mux",       "04", "MUX switch matrix"),
    ("power",     "05", "Power management (IDAC, PCA9535)"),
    ("usbpd",     "06", "USB Power Delivery"),
    ("wavegen",   "07", "Waveform generator"),
    ("wifi",      "08", "WiFi management"),
    ("selftest",  "09", "Self-test & calibration"),
    ("streaming", "10", "ADC/scope streaming (USB only)"),
    ("hat",       "11", "HAT expansion board"),
    ("faults",    "12", "Fault management"),
    ("http",      None, "HTTP REST endpoints"),
]

# ── Connection panel widgets ──────────────────────────────────────────────────
_ports_now = detect_serial_ports()
_port_opts  = ["(none)"] + _ports_now

w_usb = widgets.Dropdown(
    options=_port_opts,
    value="(none)",
    description="USB Port:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "80px"},
)
w_usb_refresh = widgets.Button(
    description="\u27f3",
    layout=widgets.Layout(width="36px", height="32px"),
    tooltip="Refresh serial port list",
)
w_http = widgets.Text(
    placeholder="192.168.4.1",
    description="HTTP IP:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "80px"},
)
w_transport = widgets.RadioButtons(
    options=["USB + HTTP", "USB only", "HTTP only"],
    value="USB + HTTP",
    description="Transport:",
    layout=widgets.Layout(width="320px"),
    style={"description_width": "80px"},
)

# ── Category panel widgets ────────────────────────────────────────────────────
_cat_options = [
    (f"{('['+n+'] ' if n else '[--] ')}{k:<10}  {desc}", k)
    for k, n, desc in CATEGORIES
]
w_cats = widgets.SelectMultiple(
    options=_cat_options,
    value=[k for k, _, _ in CATEGORIES],
    rows=13,
    layout=widgets.Layout(width="420px", font_family="monospace"),
)
w_all_btn  = widgets.Button(description="Select All", layout=widgets.Layout(width="110px"))
w_none_btn = widgets.Button(description="Clear",      layout=widgets.Layout(width="80px"))

# ── Options panel widgets ─────────────────────────────────────────────────────
w_hat            = widgets.Checkbox(value=False, description="HAT board connected",  indent=False)
w_no_destructive = widgets.Checkbox(value=True,  description="Skip destructive tests", indent=False)
w_stop_first     = widgets.Checkbox(value=False, description="Stop on first failure",  indent=False)
w_timeout = widgets.IntSlider(
    min=10, max=120, step=5, value=30,
    description="Timeout (s):",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "90px"},
)
w_verbose = widgets.Checkbox(value=False, description="Verbose output", indent=False)

# ── Action buttons ────────────────────────────────────────────────────────────
w_run = widgets.Button(
    description="\u25b6  Run Tests",
    button_style="success",
    layout=widgets.Layout(width="160px", height="44px"),
)
w_open_report = widgets.Button(
    description="\U0001f4c4 Open HTML Report",
    button_style="info",
    layout=widgets.Layout(width="190px", height="44px"),
)
w_clear = widgets.Button(
    description="\U0001f5d1  Clear",
    button_style="warning",
    layout=widgets.Layout(width="110px", height="44px"),
)

# ── Output areas ──────────────────────────────────────────────────────────────
out_status = widgets.Output()
out_live   = widgets.Output(
    layout=widgets.Layout(
        max_height="350px", overflow_y="auto",
        background="#0d1117", border="1px solid #30363d",
        border_radius="6px", padding="0",
        font_family="monospace", font_size="12px",
    )
)
out_charts = widgets.Output()
out_table  = widgets.Output()

# ── Button callbacks ──────────────────────────────────────────────────────────
def refresh_ports(b):
    fresh = ["(none)"] + detect_serial_ports()
    old_val = w_usb.value
    w_usb.options = fresh
    if old_val in fresh:
        w_usb.value = old_val

def select_all(b):
    w_cats.value = [k for k, _, _ in CATEGORIES]

def select_none(b):
    w_cats.value = []

w_usb_refresh.on_click(refresh_ports)
w_all_btn.on_click(select_all)
w_none_btn.on_click(select_none)

# ── Layout assembly ───────────────────────────────────────────────────────────
_header_html = widgets.HTML("""
<div style="background:linear-gradient(135deg,#0d1b2a,#1e3a5f);padding:18px 22px;
            border-radius:10px;margin-bottom:16px;border:1px solid #30363d">
  <h2 style="color:#58a6ff;margin:0;font-family:monospace;letter-spacing:1px">
    \U0001f52c BugBuster Test Dashboard</h2>
  <p style="color:#8b949e;margin:6px 0 0;font-size:13px">
    Hardware-in-the-loop &middot; ESP32 + RP2040 &middot; Python API &middot; HTTP REST &middot; Desktop App</p>
</div>
""")

_conn_label    = widgets.HTML("<b style='color:#8b949e;font-size:12px;letter-spacing:1px'>CONNECTION</b>")
_options_label = widgets.HTML("<b style='color:#8b949e;font-size:12px;letter-spacing:1px'>OPTIONS</b>")
_cats_label    = widgets.HTML("<b style='color:#8b949e;font-size:12px;letter-spacing:1px'>CATEGORIES</b>")
_cats_btns     = widgets.HBox([w_all_btn, w_none_btn])

_usb_row = widgets.HBox(
    [w_usb, w_usb_refresh],
    layout=widgets.Layout(align_items="center"),
)

_left_panel = widgets.VBox(
    [
        _conn_label,
        _usb_row,
        w_http,
        w_transport,
        widgets.HTML("<hr style='border-color:#30363d;margin:10px 0'>"),
        _options_label,
        w_hat,
        w_no_destructive,
        w_stop_first,
        w_timeout,
        w_verbose,
    ],
    layout=widgets.Layout(width="380px", padding="0 16px 0 0"),
)

_right_panel = widgets.VBox(
    [_cats_label, w_cats, _cats_btns],
    layout=widgets.Layout(width="440px"),
)

_top_row = widgets.HBox(
    [_left_panel, _right_panel],
    layout=widgets.Layout(align_items="flex-start"),
)

_action_row = widgets.HBox(
    [w_run, w_open_report, w_clear, out_status],
    layout=widgets.Layout(align_items="center", gap="10px", margin="10px 0"),
)


# ── Desktop App test widgets ──────────────────────────────────────────────────
DESKTOP_APP_DIR = REPO_ROOT / "DesktopApp" / "BugBuster"
TAURI_DIR       = DESKTOP_APP_DIR / "src-tauri"
E2E_DIR         = DESKTOP_APP_DIR / "tests" / "e2e"

w_rust_tests = widgets.Button(
    description="⚙  Rust Unit Tests",
    button_style="primary",
    tooltip="Run cargo test in src-tauri/ (no device required)",
    layout=widgets.Layout(width="190px", height="44px"),
)
w_cargo_check = widgets.Button(
    description="🔍  cargo check",
    button_style="",
    tooltip="Quick compile check — no tests, just type-check",
    layout=widgets.Layout(width="155px", height="44px"),
)
w_e2e_tests = widgets.Button(
    description="🖥  E2E Tests",
    button_style="primary",
    tooltip="Run npm test in tests/e2e/ (needs tauri-driver + compiled app)",
    layout=widgets.Layout(width="160px", height="44px"),
)

out_rust_status  = widgets.Output()
out_rust_live    = widgets.Output(layout=widgets.Layout(
    max_height="280px", overflow_y="auto",
    background="#0d1117", border="1px solid #30363d",
    border_radius="6px", padding="0",
    font_family="monospace", font_size="12px",
))
out_rust_summary = widgets.Output()
out_e2e_status   = widgets.Output()
out_e2e_live     = widgets.Output(layout=widgets.Layout(
    max_height="280px", overflow_y="auto",
    background="#0d1117", border="1px solid #30363d",
    border_radius="6px", padding="0",
    font_family="monospace", font_size="12px",
))
out_e2e_summary  = widgets.Output()

# ── Desktop section layout ────────────────────────────────────────────────────
_desktop_header = widgets.HTML("""
<div style="background:linear-gradient(135deg,#0d1b2a,#1a2f1a);padding:14px 22px;
            border-radius:10px;margin:16px 0 8px;border:1px solid #30363d">
  <h3 style="color:#3fb950;margin:0;font-family:monospace;letter-spacing:1px">
    🖥 Desktop App Tests</h3>
  <p style="color:#8b949e;margin:4px 0 0;font-size:12px">
    Rust unit tests (cargo test) &middot; E2E WebDriverIO tests (tauri-driver) — no device required</p>
</div>
""")

_desktop_rust_row = widgets.HBox(
    [w_rust_tests, w_cargo_check, out_rust_status],
    layout=widgets.Layout(align_items="center", gap="10px", margin="6px 0"),
)
_desktop_e2e_row = widgets.HBox(
    [w_e2e_tests, out_e2e_status],
    layout=widgets.Layout(align_items="center", gap="10px", margin="2px 0"),
)

_desktop_section = widgets.VBox(
    [
        _desktop_header,
        widgets.HTML("<b style=\'color:#8b949e;font-size:12px;letter-spacing:1px\'>RUST UNIT TESTS (cargo test)</b>"),
        _desktop_rust_row,
        out_rust_live,
        out_rust_summary,
        widgets.HTML("<hr style=\'border-color:#30363d;margin:12px 0\'>"),
        widgets.HTML("<b style=\'color:#8b949e;font-size:12px;letter-spacing:1px\'>E2E TESTS (WebDriverIO + tauri-driver)</b>"),
        _desktop_e2e_row,
        out_e2e_live,
        out_e2e_summary,
    ],
    layout=widgets.Layout(padding="0"),
)

_dashboard = widgets.VBox(
    [
        _header_html,
        _top_row,
        widgets.HTML("<hr style='border-color:#30363d;margin:12px 0'>"),
        _action_row,
        out_live,
        out_charts,
        out_table,
        _desktop_section,
    ],
    layout=widgets.Layout(padding="8px"),
)

display(_dashboard)

In [ ]:
"""
Runner cell — defines the run_tests callback and wires it to the button.
Run this cell after Cell 3.
"""

STATUS_EMOJI = {"passed": "\u2705", "failed": "\u274c", "skipped": "\u23ed\ufe0f", "error": "\U0001f4a5", "unknown": "\u2753"}
STATUS_COLOR = {
    "passed":  "#3fb950",
    "failed":  "#f85149",
    "skipped": "#8b949e",
    "error":   "#d29922",
    "unknown": "#6e7681",
}

_last_results = {}   # mutable store shared between callbacks

# ── Command builder ──────────────────────────────────────────────────────────
def _build_cmd():
    cmd = [sys.executable, "-m", "pytest"]

    usb  = w_usb.value  if w_usb.value  != "(none)" else None
    http = w_http.value.strip() or None
    mode = w_transport.value   # "USB + HTTP" | "USB only" | "HTTP only"

    if usb  and "USB"  in mode: cmd.append(f"--device-usb={usb}")
    if http and "HTTP" in mode: cmd.append(f"--device-http={http}")
    if w_hat.value:              cmd.append("--hat")
    if w_no_destructive.value:   cmd.append("--skip-destructive")
    if w_stop_first.value:       cmd.append("-x")

    if   mode == "USB only":  cmd += ["-k", "not http_only"]
    elif mode == "HTTP only": cmd += ["-k", "not usb_only and not streaming"]

    # Category \u2192 file path
    cat_map = {
        k: (f"device/test_{n}_{k}.py" if n else "http_api/test_http_endpoints.py")
        for k, n, _ in CATEGORIES
    }
    selected = list(w_cats.value)
    if selected and len(selected) < len(CATEGORIES):
        for cat in selected:
            cmd.append(str(TESTS_DIR / cat_map[cat]))
    else:
        cmd.append(str(TESTS_DIR))

    report_json = TESTS_DIR / ".report.json"
    report_html = TESTS_DIR / "report.html"
    cmd += [
        f"--timeout={w_timeout.value}",
        "--tb=short",
        "--json-report", f"--json-report-file={report_json}",
        f"--html={report_html}", "--self-contained-html",
        "--no-header",
        "-v" if w_verbose.value else "-q",
    ]
    return cmd, report_json, report_html

# ── Run callback ─────────────────────────────────────────────────────────────
def on_run(b):
    w_run.disabled = True
    w_run.description = "\u23f3  Running\u2026"
    w_run.button_style = "warning"
    out_live.clear_output()
    out_charts.clear_output()
    out_table.clear_output()

    with out_status:
        clear_output(wait=True)
        display(HTML("""
        <span style="background:#d29922;color:#0d1117;padding:4px 14px;border-radius:12px;
                     font-family:monospace;font-size:13px;font-weight:bold">\u23f3 Tests running\u2026</span>
        """))

    cmd, report_json, report_html = _build_cmd()

    with out_live:
        display(HTML(f"""
        <div style="background:#0d1117;color:#58a6ff;padding:8px 12px;
                    font-family:monospace;font-size:12px;border-radius:4px 4px 0 0;
                    border-bottom:1px solid #30363d">
          $ pytest {" ".join(str(a) for a in cmd[3:])}
        </div>
        """))

        proc = subprocess.Popen(
            cmd, cwd=str(REPO_ROOT),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        buf = []
        for line in proc.stdout:
            buf.append(line)
            # Flush every 10 lines for live feel
            if len(buf) >= 10:
                print("".join(buf), end="", flush=True)
                buf = []
        if buf:
            print("".join(buf), end="", flush=True)
        proc.wait()

    # Load JSON report
    if report_json.exists():
        with open(report_json) as f:
            data = json.load(f)
        _last_results.update(data)
        _render_results(data, report_html)

        summary = data.get("summary", {})
        failed  = summary.get("failed", 0)
        passed  = summary.get("passed", 0)
        total   = summary.get("total",  0)

        if failed == 0 and total > 0:
            badge = (f'<span style="background:#3fb950;color:#0d1117;padding:4px 14px;'
                     f'border-radius:12px;font-family:monospace;font-size:13px;font-weight:bold">'
                     f'\u2705  All {passed} tests passed</span>')
        elif failed > 0:
            badge = (f'<span style="background:#f85149;color:#fff;padding:4px 14px;'
                     f'border-radius:12px;font-family:monospace;font-size:13px;font-weight:bold">'
                     f'\u274c  {failed} of {total} tests FAILED</span>')
        else:
            badge = (f'<span style="background:#8b949e;color:#0d1117;padding:4px 14px;'
                     f'border-radius:12px;font-family:monospace;font-size:13px;font-weight:bold">'
                     f'\u23ed\ufe0f  No tests ran</span>')

        with out_status:
            clear_output(wait=True)
            display(HTML(badge))
    else:
        with out_status:
            clear_output(wait=True)
            display(HTML('<span style="background:#f85149;color:#fff;padding:4px 14px;'
                         'border-radius:12px;font-family:monospace">\U0001f4a5 pytest failed to produce a report</span>'))

    w_run.disabled = False
    w_run.description = "\u25b6  Run Tests"
    w_run.button_style = "success"


def on_clear(b):
    out_live.clear_output()
    out_charts.clear_output()
    out_table.clear_output()
    with out_status:
        clear_output(wait=True)


def on_open_report(b):
    _, _, report_html = _build_cmd()
    if report_html.exists():
        import platform
        opener = "open" if platform.system() == "Darwin" else "xdg-open"
        subprocess.Popen([opener, str(report_html)])
    else:
        with out_status:
            clear_output(wait=True)
            display(HTML('<span style="color:#f85149">No report.html yet \u2014 run tests first.</span>'))


w_run.on_click(on_run)
w_clear.on_click(on_clear)
w_open_report.on_click(on_open_report)
print("\u2705  Runner wired \u2014 click 'Run Tests' in the UI above.")

# =============================================================================
# Desktop App Test Callbacks
# =============================================================================

def _parse_rust_results(output: str):
    """Parse cargo test stdout and return (passed, failed, ignored) counts."""
    import re
    m = re.search(r'test result:.*?([0-9]+) passed; ([0-9]+) failed; ([0-9]+) ignored', output)
    if m:
        return int(m.group(1)), int(m.group(2)), int(m.group(3))
    return None, None, None

def _run_cmd_streamed(cmd, cwd, out_live_widget, out_status_widget, label):
    """Run a subprocess, stream output, return (returncode, full_output)."""
    with out_status_widget:
        clear_output(wait=True)
        display(HTML(f'''
        <span style="background:#d29922;color:#0d1117;padding:3px 12px;border-radius:12px;
                     font-family:monospace;font-size:12px;font-weight:bold">⏳ {label} running…</span>
        '''))

    with out_live_widget:
        clear_output(wait=True)
        display(HTML(f'''
        <div style="background:#0d1117;color:#58a6ff;padding:6px 12px;
                    font-family:monospace;font-size:12px;border-radius:4px 4px 0 0;
                    border-bottom:1px solid #30363d">
          $ {" ".join(str(a) for a in cmd)}
        </div>
        '''))

        proc = subprocess.Popen(
            cmd, cwd=str(cwd),
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        buf = []
        full_output = []
        for line in proc.stdout:
            buf.append(line)
            full_output.append(line)
            if len(buf) >= 5:
                print("".join(buf), end="", flush=True)
                buf = []
        if buf:
            print("".join(buf), end="", flush=True)
        proc.wait()
        return proc.returncode, "".join(full_output)

def on_rust_tests(b):
    import shutil
    w_rust_tests.disabled = True
    w_cargo_check.disabled = True
    out_rust_summary.clear_output()

    cargo = shutil.which("cargo")
    if not cargo:
        with out_rust_status:
            clear_output(wait=True)
            display(HTML('<span style="background:#f85149;color:#fff;padding:3px 12px;border-radius:12px;font-family:monospace;font-size:12px">💥 cargo not found in PATH</span>'))
        w_rust_tests.disabled = False
        w_cargo_check.disabled = False
        return

    cmd = [cargo, "test", "--color=always"]
    rc, out = _run_cmd_streamed(cmd, TAURI_DIR, out_rust_live, out_rust_status, "cargo test")

    passed, failed, ignored = _parse_rust_results(out)

    with out_rust_summary:
        clear_output(wait=True)
        if rc == 0 and passed is not None:
            display(HTML(f'''
            <div style="display:flex;gap:10px;margin-top:8px;align-items:center">
              <span style="background:#3fb950;color:#0d1117;padding:4px 14px;border-radius:12px;
                           font-family:monospace;font-size:13px;font-weight:bold">
                ✅  {passed} tests passed</span>
              {f'<span style="background:#8b949e;color:#0d1117;padding:4px 10px;border-radius:12px;font-family:monospace;font-size:12px">{ignored} ignored</span>' if ignored else ""}
            </div>
            '''))
        elif rc != 0 and failed is not None:
            display(HTML(f'''
            <div style="display:flex;gap:10px;margin-top:8px;align-items:center">
              <span style="background:#f85149;color:#fff;padding:4px 14px;border-radius:12px;
                           font-family:monospace;font-size:13px;font-weight:bold">
                ❌  {failed} failed, {passed} passed</span>
            </div>
            '''))
        else:
            display(HTML('<span style="background:#f85149;color:#fff;padding:4px 14px;border-radius:12px;font-family:monospace;font-size:12px">💥 cargo test failed (no result summary found)</span>'))

    with out_rust_status:
        clear_output(wait=True)
        if rc == 0:
            display(HTML('<span style="background:#3fb950;color:#0d1117;padding:3px 12px;border-radius:12px;font-family:monospace;font-size:12px;font-weight:bold">✅ PASS</span>'))
        else:
            display(HTML('<span style="background:#f85149;color:#fff;padding:3px 12px;border-radius:12px;font-family:monospace;font-size:12px;font-weight:bold">❌ FAIL</span>'))

    w_rust_tests.disabled = False
    w_cargo_check.disabled = False


def on_cargo_check(b):
    import shutil
    w_rust_tests.disabled = True
    w_cargo_check.disabled = True
    out_rust_summary.clear_output()

    cargo = shutil.which("cargo")
    if not cargo:
        with out_rust_status:
            clear_output(wait=True)
            display(HTML('<span style="background:#f85149;color:#fff;padding:3px 12px;border-radius:12px;font-family:monospace;font-size:12px">💥 cargo not found</span>'))
        w_rust_tests.disabled = False
        w_cargo_check.disabled = False
        return

    cmd = [cargo, "check", "--color=always"]
    rc, _ = _run_cmd_streamed(cmd, TAURI_DIR, out_rust_live, out_rust_status, "cargo check")

    with out_rust_status:
        clear_output(wait=True)
        if rc == 0:
            display(HTML('<span style="background:#3fb950;color:#0d1117;padding:3px 12px;border-radius:12px;font-family:monospace;font-size:12px;font-weight:bold">✅ OK</span>'))
        else:
            display(HTML('<span style="background:#f85149;color:#fff;padding:3px 12px;border-radius:12px;font-family:monospace;font-size:12px;font-weight:bold">❌ Errors</span>'))

    w_rust_tests.disabled = False
    w_cargo_check.disabled = False


def on_e2e_tests(b):
    import shutil
    w_e2e_tests.disabled = True
    out_e2e_summary.clear_output()

    # Check prerequisites
    if not E2E_DIR.exists():
        with out_e2e_status:
            clear_output(wait=True)
            display(HTML(f'<span style="background:#f85149;color:#fff;padding:3px 12px;border-radius:12px;font-family:monospace;font-size:12px">💥 E2E dir not found: {E2E_DIR}</span>'))
        w_e2e_tests.disabled = False
        return

    node_modules = E2E_DIR / "node_modules"
    if not node_modules.exists():
        with out_e2e_status:
            clear_output(wait=True)
            display(HTML('<span style="background:#d29922;color:#0d1117;padding:3px 12px;border-radius:12px;font-family:monospace;font-size:12px">⚙ Installing npm deps…</span>'))
        with out_e2e_live:
            clear_output(wait=True)
        npm = shutil.which("npm")
        if npm:
            subprocess.run([npm, "install"], cwd=str(E2E_DIR), capture_output=True)

    app_binary = DESKTOP_APP_DIR / "target" / "debug" / "bugbuster"
    if not app_binary.exists():
        with out_e2e_summary:
            display(HTML(f'''
            <div style="background:#1c1c24;border:1px solid #d29922;border-radius:6px;padding:10px 14px;margin-top:8px;font-family:monospace;font-size:12px;color:#d29922">
              ⚠ App binary not found at <code>{app_binary}</code>.<br>
              Build first: <code>cd DesktopApp/BugBuster && cargo build</code>
            </div>
            '''))

    npm = shutil.which("npm")
    if not npm:
        with out_e2e_status:
            clear_output(wait=True)
            display(HTML('<span style="background:#f85149;color:#fff;padding:3px 12px;border-radius:12px;font-family:monospace;font-size:12px">💥 npm not found in PATH</span>'))
        w_e2e_tests.disabled = False
        return

    cmd = [npm, "test"]
    rc, out = _run_cmd_streamed(cmd, E2E_DIR, out_e2e_live, out_e2e_status, "E2E tests")

    # Parse wdio output: look for "X passing", "X failing"
    import re
    passing = sum(int(m) for m in re.findall(r"(\d+) passing", out))
    failing  = sum(int(m) for m in re.findall(r"(\d+) failing", out))

    with out_e2e_summary:
        clear_output(wait=True)
        if rc == 0 or (passing > 0 and failing == 0):
            display(HTML(f'''
            <div style="display:flex;gap:10px;margin-top:8px;align-items:center">
              <span style="background:#3fb950;color:#0d1117;padding:4px 14px;border-radius:12px;
                           font-family:monospace;font-size:13px;font-weight:bold">
                ✅  {passing} E2E tests passed</span>
            </div>
            '''))
        elif failing > 0:
            display(HTML(f'''
            <div style="display:flex;gap:10px;margin-top:8px;align-items:center">
              <span style="background:#f85149;color:#fff;padding:4px 14px;border-radius:12px;
                           font-family:monospace;font-size:13px;font-weight:bold">
                ❌  {failing} failed, {passing} passed</span>
            </div>
            '''))
        else:
            tip = "tauri-driver not installed?" if "tauri-driver" in out.lower() or "not found" in out.lower() else "Check output above"
            display(HTML(f'<span style="background:#f85149;color:#fff;padding:4px 14px;border-radius:12px;font-family:monospace;font-size:12px">💥 E2E run failed — {tip}</span>'))

    with out_e2e_status:
        clear_output(wait=True)
        if rc == 0:
            display(HTML('<span style="background:#3fb950;color:#0d1117;padding:3px 12px;border-radius:12px;font-family:monospace;font-size:12px;font-weight:bold">✅ PASS</span>'))
        else:
            display(HTML('<span style="background:#f85149;color:#fff;padding:3px 12px;border-radius:12px;font-family:monospace;font-size:12px;font-weight:bold">❌ FAIL</span>'))

    w_e2e_tests.disabled = False


w_rust_tests.on_click(on_rust_tests)
w_cargo_check.on_click(on_cargo_check)
w_e2e_tests.on_click(on_e2e_tests)


In [ ]:
"""
Results rendering functions — called automatically after tests complete.
You can also call _render_results(_last_results, ...) manually to re-render.
"""

def _make_donut(passed, failed, skipped, errors):
    labels = ["Passed", "Failed", "Skipped", "Error"]
    values = [passed, failed, skipped, errors]
    colors = [STATUS_COLOR["passed"], STATUS_COLOR["failed"],
              STATUS_COLOR["skipped"], STATUS_COLOR["error"]]
    total  = sum(values)
    pct    = round(passed / total * 100) if total else 0
    banner = STATUS_COLOR["passed"] if failed == 0 and total > 0 else STATUS_COLOR["failed"]

    fig = go.Figure(go.Pie(
        labels=labels, values=values, hole=0.65,
        marker_colors=colors, textinfo="label+percent",
        hovertemplate="%{label}: %{value}<extra></extra>",
        sort=False,
    ))
    fig.update_layout(
        paper_bgcolor="#161b22", plot_bgcolor="#161b22",
        font=dict(color="#c9d1d9", size=12),
        showlegend=False, width=270, height=250,
        margin=dict(l=10, r=10, t=30, b=10),
        title=dict(text="Overall", font=dict(color="#8b949e", size=12), x=0.5),
        annotations=[dict(text=f"<b>{pct}%</b>", x=0.5, y=0.5, showarrow=False,
                          font=dict(size=24, color=banner, family="monospace"))],
    )
    return fig


def _make_category_bar(df):
    cats_in_order = [k for k, _, _ in CATEGORIES if k in df["category"].values]
    cat_df = (df.groupby(["category", "status"])
                .size().rename("count").reset_index())
    fig = go.Figure()
    for status in ["passed", "failed", "skipped", "error"]:
        sub = cat_df[cat_df["status"] == status]
        sub = sub.set_index("category").reindex(cats_in_order, fill_value=0).reset_index()
        fig.add_trace(go.Bar(
            name=status.capitalize(),
            x=sub["count"].tolist(),
            y=sub["category"].tolist(),
            orientation="h",
            marker_color=STATUS_COLOR[status],
            hovertemplate=f"%{{y}}: %{{x}} {status}<extra></extra>",
        ))
    fig.update_layout(
        barmode="stack",
        paper_bgcolor="#161b22", plot_bgcolor="#1c2128",
        font=dict(color="#c9d1d9", size=12),
        xaxis=dict(gridcolor="#30363d", title="Tests"),
        yaxis=dict(gridcolor="#30363d"),
        legend=dict(orientation="h", y=1.06, x=0, font=dict(size=11)),
        height=max(300, 28 * len(cats_in_order) + 90),
        width=550,
        margin=dict(l=10, r=20, t=50, b=30),
        title=dict(text="Results by Category", font=dict(color="#8b949e", size=12)),
    )
    return fig


def _make_duration_bar(df):
    top = df.nlargest(20, "duration")[["name", "duration", "status", "category"]].copy()
    fig = px.bar(
        top, x="duration", y="name", orientation="h",
        color="status",
        color_discrete_map=STATUS_COLOR,
        title="20 Slowest Tests",
        labels={"duration": "Duration (s)", "name": ""},
    )
    fig.update_layout(
        paper_bgcolor="#161b22", plot_bgcolor="#1c2128",
        font=dict(color="#c9d1d9", size=11),
        xaxis=dict(gridcolor="#30363d"),
        yaxis=dict(gridcolor="#30363d", autorange="reversed"),
        showlegend=False,
        height=max(280, 22 * len(top) + 80),
        width=700,
        margin=dict(l=10, r=20, t=40, b=30),
        title=dict(text="20 Slowest Tests", font=dict(color="#8b949e", size=12)),
    )
    return fig


def _make_transport_bar(df):
    tr_df = (df.groupby(["transport", "status"])
               .size().rename("count").reset_index())
    transports = sorted(df["transport"].unique())
    fig = go.Figure()
    for status in ["passed", "failed", "skipped"]:
        sub = tr_df[tr_df["status"] == status].set_index("transport")
        counts = [int(sub.loc[t, "count"]) if t in sub.index else 0 for t in transports]
        fig.add_trace(go.Bar(
            name=status.capitalize(), x=transports, y=counts,
            marker_color=STATUS_COLOR[status],
        ))
    fig.update_layout(
        barmode="stack",
        paper_bgcolor="#161b22", plot_bgcolor="#1c2128",
        font=dict(color="#c9d1d9", size=11),
        xaxis=dict(gridcolor="#30363d"),
        yaxis=dict(gridcolor="#30363d"),
        showlegend=False, height=220, width=260,
        margin=dict(l=20, r=10, t=40, b=30),
        title=dict(text="By Transport", font=dict(color="#8b949e", size=12)),
    )
    return fig


def _parse_df(data):
    """Parse pytest JSON report into a DataFrame."""
    rows = []
    cat_lookup = {k: k for k, _, _ in CATEGORIES}
    for t in data.get("tests", []):
        nodeid = t.get("nodeid", "")
        parts  = nodeid.split("::")
        file_p = parts[0] if parts else ""
        cat = "unknown"
        for k in cat_lookup:
            if k in file_p:
                cat = k
                break

        transport = "usb" if "[usb" in nodeid else ("http" if "[http" in nodeid else "\u2014")
        status    = t.get("outcome", "unknown")
        dur       = t.get("duration") or 0
        call      = t.get("call") or {}
        message   = call.get("longrepr", "") if status in ("failed", "error") else ""

        rows.append({
            "status":    status,
            "category":  cat,
            "transport": transport,
            "name":      parts[-1] if len(parts) > 1 else nodeid,
            "nodeid":    nodeid,
            "duration":  dur,
            "message":   str(message)[:4000],
        })
    return pd.DataFrame(rows)


def _render_results(data, report_html=None):
    summary  = data.get("summary", {})
    passed   = summary.get("passed",  0)
    failed   = summary.get("failed",  0)
    skipped  = summary.get("skipped", 0)
    errors   = summary.get("error",   0)
    total    = summary.get("total",   0)
    duration = data.get("duration",   0)
    pass_pct = round(passed / total * 100) if total else 0
    bcolor   = STATUS_COLOR["passed"] if failed == 0 and total > 0 else STATUS_COLOR["failed"]

    df = _parse_df(data)

    # ── Summary numbers ──────────────────────────────────────────────────────
    with out_charts:
        clear_output(wait=True)

        _stat_items = "".join(
            f'<div style="text-align:center;min-width:56px">'
            f'<div style="font-size:30px;font-weight:bold;color:{STATUS_COLOR[k]};font-family:monospace">{v}</div>'
            f'<div style="color:#8b949e;font-size:11px;text-transform:uppercase;letter-spacing:1px">{k}</div>'
            f'</div>'
            for k, v in [("passed", passed), ("failed", failed), ("skipped", skipped), ("error", errors)]
        )
        _report_link = (
            f'<a href="{report_html}" target="_blank" style="color:#58a6ff;font-size:12px;'
            f'text-decoration:none;border:1px solid #30363d;padding:4px 10px;border-radius:6px">'
            f'\U0001f4c4 Full Report</a>'
            if report_html is not None and Path(str(report_html)).exists() else ""
        )

        display(HTML(f"""
        <div style="background:#161b22;border:1px solid #30363d;border-radius:8px;
                    padding:16px 22px;margin:10px 0;display:flex;gap:28px;
                    align-items:center;flex-wrap:wrap">
          <div style="text-align:center">
            <div style="font-size:40px;font-weight:bold;color:{bcolor};font-family:monospace">{pass_pct}%</div>
            <div style="color:#8b949e;font-size:11px;text-transform:uppercase;letter-spacing:1px">Pass Rate</div>
          </div>
          <div style="width:1px;height:54px;background:#30363d"></div>
          {_stat_items}
          <div style="width:1px;height:54px;background:#30363d"></div>
          <div style="text-align:center;min-width:56px">
            <div style="font-size:30px;font-weight:bold;color:#58a6ff;font-family:monospace">{total}</div>
            <div style="color:#8b949e;font-size:11px;text-transform:uppercase;letter-spacing:1px">total</div>
          </div>
          <div style="text-align:center;min-width:56px">
            <div style="font-size:30px;font-weight:bold;color:#8b949e;font-family:monospace">{duration:.1f}s</div>
            <div style="color:#8b949e;font-size:11px;text-transform:uppercase;letter-spacing:1px">time</div>
          </div>
          {_report_link}
        </div>
        """))

        if df.empty:
            display(HTML("<p style='color:#8b949e;font-style:italic'>No test data.</p>"))
            return

        # ── Charts row ───────────────────────────────────────────────────────
        _make_donut(passed, failed, skipped, errors).show()
        _make_category_bar(df).show()
        if df["transport"].nunique() > 1:
            _make_transport_bar(df).show()
        if df["duration"].sum() > 0.1:
            _make_duration_bar(df).show()

    # ── Detail table + failure drill-down ────────────────────────────────────
    with out_table:
        clear_output(wait=True)

        display(HTML("""
        <style>
        .bb-table{border-collapse:collapse;width:100%;font-family:monospace;font-size:12px}
        .bb-table th{background:#21262d;color:#8b949e;padding:7px 12px;text-align:left;
                     border-bottom:2px solid #30363d;position:sticky;top:0}
        .bb-table td{padding:5px 12px;border-bottom:1px solid #21262d;color:#c9d1d9;
                     white-space:nowrap;max-width:340px;overflow:hidden;text-overflow:ellipsis}
        .bb-table tr:hover td{background:#1c2128}
        details>summary{cursor:pointer;padding:4px 8px;color:#f85149;font-family:monospace;font-size:12px}
        details pre{font-size:11px;color:#c9d1d9;background:#0d1117;
                    padding:8px;border-radius:4px;white-space:pre-wrap;
                    max-height:200px;overflow-y:auto;margin:4px 0}
        </style>
        """))

        # ── Interactive filter ───────────────────────────────────────────────
        w_search  = widgets.Text(placeholder="filter by name / category\u2026",
                                 layout=widgets.Layout(width="360px"))
        w_statflt = widgets.SelectMultiple(
            options=["passed", "failed", "skipped", "error"],
            value=["passed", "failed", "skipped", "error"],
            rows=4, layout=widgets.Layout(width="110px"),
        )
        filter_out = widgets.Output()

        def _refresh_table(*_):
            q  = w_search.value.lower()
            st = set(w_statflt.value)
            mask = df["status"].isin(st)
            if q:
                mask &= (df["name"].str.lower().str.contains(q, regex=False) |
                         df["category"].str.lower().str.contains(q, regex=False))
            sub = df[mask].copy()

            rows_html = ""
            for _, r in sub.iterrows():
                em = STATUS_EMOJI.get(r["status"], "\u2753")
                cl = STATUS_COLOR.get(r["status"], "#6e7681")
                rows_html += (
                    f"<tr>"
                    f"<td style='color:{cl}'>{em} {r['status']}</td>"
                    f"<td>{r['category']}</td>"
                    f"<td>{r['transport']}</td>"
                    f"<td title='{r['nodeid']}'>{r['name']}</td>"
                    f"<td style='color:#8b949e'>{r['duration']:.3f}s</td>"
                    f"</tr>"
                )

            table_html = f"""
            <div style="max-height:420px;overflow-y:auto;border:1px solid #30363d;border-radius:6px">
            <table class="bb-table">
              <thead><tr>
                <th>Status</th><th>Category</th><th>Transport</th>
                <th>Test</th><th>Duration</th>
              </tr></thead>
              <tbody>{rows_html}</tbody>
            </table>
            </div>
            <p style="color:#6e7681;font-size:11px;margin:4px 0">{len(sub)} / {len(df)} tests shown</p>
            """

            # Failures drill-down
            fail_df = sub[sub["status"].isin(["failed", "error"])]
            fail_html = ""
            if not fail_df.empty:
                fail_html = "<h4 style='color:#f85149;margin:16px 0 8px;font-family:monospace'>\u274c Failures</h4>"
                for _, r in fail_df.iterrows():
                    msg = r["message"].replace("<", "&lt;").replace(">", "&gt;") if r["message"] else "(no message)"
                    fail_html += f"""
                    <details style="margin:4px 0;background:#1c2128;border:1px solid {STATUS_COLOR['failed']};
                                    border-radius:5px;padding:0 8px 4px">
                      <summary>{r['nodeid']}</summary>
                      <pre>{msg[:3000]}</pre>
                    </details>"""

            with filter_out:
                clear_output(wait=True)
                display(HTML(table_html + fail_html))

        w_search.observe(_refresh_table, names="value")
        w_statflt.observe(_refresh_table, names="value")

        display(widgets.HBox([
            widgets.VBox([widgets.HTML("<b style='color:#8b949e;font-size:12px'>SEARCH</b>"), w_search]),
            widgets.VBox([widgets.HTML("<b style='color:#8b949e;font-size:12px'>STATUS</b>"),  w_statflt]),
        ]))
        display(filter_out)
        _refresh_table()


print("\u2705  Renderer defined")

In [ ]:
# Re-render last results (useful if you closed outputs by accident)
if _last_results:
    _, _, rhtml = _build_cmd()
    _render_results(_last_results, rhtml)
else:
    print("No results yet \u2014 run tests first.")

---
## Tips

- **Run cells in order**: 2 → 3 → 4 → 5. Then use the UI to run tests.
- **Filter categories**: Hold ⌘/Ctrl to select multiple in the category list.
- **Re-render**: If charts disappear, run Cell 6 to redraw the last results.
- **Terminal alternative**: `python tests/run_tests.py --usb /dev/cu.usbmodem1234`
- **HTML report**: Click "📄 Open HTML Report" after a test run for the full pytest-html report.